# ALICE Analysis: YFV19 and Ankylosing Spondylitis (AS) TCR-seq datasets

This notebook runs the ALICE (Antigen-specific Lymphocyte Identification by Clustering of
Expanded CDR3s) algorithm on two benchmark datasets from `isalgo/airr_benchmark`:

- **YF (Yellow Fever vaccine)** — 6 donors × 2 time points (day 0, day 15) TRB repertoires.
- **AS (Ankylosing Spondylitis)** — 4 donors, synovial fluid CD8+ TRB repertoires.

Outline:
1. Environment setup and dataset download
2. Parse and validate YF samples
3. Run ALICE on YF samples, collect hits (Poisson FDR < 0.05)
4. Bar plot: ALICE hit counts per donor, day 0 vs day 15
5. Filter ALICE hits to HLA-A*02 LLWNGPMAV-specific clonotypes (VDJdb); re-plot
6. Parse AS samples
7. Run ALICE on AS samples
8. Build Hamming clonotype graph for AS ALICE hits (via `mir.graph.edit_distance_graph`)
9. Render graph with donor-color nodes, multi-donor pie fills, highlighted target clonotypes

In [ ]:
"""Cell 1: Environment setup, imports, and HF dataset download."""
import sys
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mir.biomarkers.alice import alice_table, compute_alice
from mir.common.filter import filter_functional
from mir.common.parser import ClonotypeTableParser, load_vdjdb_latest
from mir.common.repertoire import LocusRepertoire, infer_locus
from mir.graph.edit_distance_graph import build_edit_distance_graph
from mir.utils.notebook_assets import ensure_airr_benchmark_alice

SEED = 42
N_JOBS = 4
FDR_THRESH = 0.05

# Download alice/yf/ and alice/as/ from isalgo/airr_benchmark HF dataset.
BENCHMARK_DIR = ensure_airr_benchmark_alice(repo_root=repo_root, subsets=["yf", "as"])
YF_DIR = BENCHMARK_DIR / "alice" / "yf"
AS_DIR = BENCHMARK_DIR / "alice" / "as"

print(f"BENCHMARK_DIR = {BENCHMARK_DIR}")
print(f"YF_DIR = {YF_DIR}  (exists: {YF_DIR.exists()})")
print(f"AS_DIR = {AS_DIR}  (exists: {AS_DIR.exists()})")
print(f"YF files: {sorted(p.name for p in YF_DIR.glob('*')) if YF_DIR.exists() else 'n/a'}")
print(f"AS files: {sorted(p.name for p in AS_DIR.glob('*')) if AS_DIR.exists() else 'n/a'}")

## 2. Parse YF samples

Files follow the MiXCR v2/v3 naming convention: `<donor>_d<day>.tsv.gz`
(e.g. `P1_d0.tsv.gz`, `S2_d15.tsv.gz`).

`ClonotypeTableParser` auto-maps MiXCR column names via `_VDJTOOLS_TO_AIRR` and
infers locus from the `j_gene` prefix — no notebook-level workarounds needed.

In [ ]:
"""Cell 2: Parse YF samples from MiXCR TSV.gz files."""

# YF donors in the benchmark dataset (confirmed by file listing).
YF_DONORS = ["P1", "P2", "Q1", "Q2", "S1", "S2"]

parser = ClonotypeTableParser()
warnings.filterwarnings("ignore", category=FutureWarning)


def _parse_yf_filename(path: Path) -> tuple[str, int]:
    """Return (donor, day) from `<donor>_d<day>.tsv.gz`."""
    stem = path.name.replace(".tsv.gz", "").replace(".tsv", "")
    donor, day_part = stem.split("_d")
    return donor, int(day_part)


yf_samples: dict[tuple[str, int], LocusRepertoire] = {}

for fp in sorted(YF_DIR.glob("*.tsv.gz")):
    try:
        donor, day = _parse_yf_filename(fp)
    except ValueError:
        print(f"  Skipping unexpected filename: {fp.name}")
        continue

    clonotypes = parser.parse(str(fp))
    # Locus is inferred from j_gene prefix in ClonotypeTableParser — no manual fix needed.
    trb = [c for c in clonotypes if c.locus == "TRB"]
    rep = filter_functional(LocusRepertoire(clonotypes=trb, locus="TRB", repertoire_id=fp.stem))
    yf_samples[(donor, day)] = rep
    print(f"  {fp.name}: {rep.clonotype_count:,} TRB clonotypes")

print(f"\nLoaded {len(yf_samples)} YF samples")

## 3. Run ALICE on YF samples

ALICE scores each clonotype by its observed neighborhood size $n$ vs the Poisson
expectation $N \cdot P_{gen}$, conditioned on the V+J gene pair (`match_mode="vj"`).
Benjamini-Hochberg FDR is applied per sample; hits = clonotypes with $q < 0.05$.

In [ ]:
"""Cell 3: Run ALICE on YF samples, collect hits at FDR < 0.05."""

from scipy.stats import false_discovery_control  # SciPy >= 1.11


def bh_fdr_col(pvals: np.ndarray) -> np.ndarray:
    """BH-corrected q-values compatible with SciPy < 1.11."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = np.clip(q, 0.0, 1.0)
    return out


yf_alice_hits: dict[tuple[str, int], pd.DataFrame] = {}
yf_alice_rows = []

for (donor, day), rep in sorted(yf_samples.items()):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = compute_alice(
            rep,
            species="human",
            threshold=1,
            match_mode="vj",
            pgen_mode="exact",
            as_table=True,
            n_jobs=N_JOBS,
            random_seed=SEED,
        )

    tbl = result.table.copy()
    tbl["q_value"] = bh_fdr_col(tbl["p_value"].values)
    hits = tbl[tbl["q_value"] < FDR_THRESH].copy()
    yf_alice_hits[(donor, day)] = hits

    yf_alice_rows.append({
        "donor": donor,
        "day": day,
        "n_clonotypes": rep.clonotype_count,
        "n_hits": len(hits),
    })
    print(f"  {donor} day{day:>3}: {rep.clonotype_count:,} clonotypes → {len(hits)} ALICE hits (FDR<0.05)")

df_yf_summary = pd.DataFrame(yf_alice_rows).sort_values(["donor", "day"]).reset_index(drop=True)
display(df_yf_summary)

## 4. Bar plot: ALICE hit counts per donor at day 0 vs day 15

In [ ]:
"""Cell 4: Bar plot of ALICE hit counts for day 0 and day 15."""

donors_ordered = sorted(df_yf_summary["donor"].unique())
days_ordered = sorted(df_yf_summary["day"].unique())
day_colors = {0: "#9ecae1", 15: "#de2d26"}
x = np.arange(len(donors_ordered))
bar_width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: raw hit counts ──────────────────────────────────────────────────────
ax = axes[0]
for i, day in enumerate(days_ordered):
    sub = df_yf_summary[df_yf_summary["day"] == day].set_index("donor")
    heights = [sub.loc[d, "n_hits"] if d in sub.index else 0 for d in donors_ordered]
    offset = (i - len(days_ordered) / 2 + 0.5) * bar_width
    ax.bar(x + offset, heights, bar_width, color=day_colors.get(day, "grey"),
           label=f"Day {day}", alpha=0.85, edgecolor="black", linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(donors_ordered)
ax.set_xlabel("Donor")
ax.set_ylabel("ALICE hits (FDR < 0.05)")
ax.set_title("ALICE hit clonotypes per donor (YF)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# ── Right: hits normalised by repertoire size ─────────────────────────────────
ax = axes[1]
for i, day in enumerate(days_ordered):
    sub = df_yf_summary[df_yf_summary["day"] == day].set_index("donor")
    heights = []
    for d in donors_ordered:
        if d in sub.index:
            n_total = max(sub.loc[d, "n_clonotypes"], 1)
            heights.append(sub.loc[d, "n_hits"] / n_total * 100)
        else:
            heights.append(0.0)
    offset = (i - len(days_ordered) / 2 + 0.5) * bar_width
    ax.bar(x + offset, heights, bar_width, color=day_colors.get(day, "grey"),
           label=f"Day {day}", alpha=0.85, edgecolor="black", linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(donors_ordered)
ax.set_xlabel("Donor")
ax.set_ylabel("ALICE hits (% of sample)")
ax.set_title("ALICE hit fraction per donor (YF)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Filter ALICE hits to HLA-A\*02 LLWNGPMAV-specific clonotypes

We intersect YF ALICE hits with VDJdb entries for LLWNGPMAV restricted to HLA-A\*02.
The intersection uses exact `junction_aa` match (which is how VDJBet also matches).
Donors with HLA-A\*02 are expected to show a strong day-15 signal.

In [ ]:
"""Cell 5: VDJdb A*02 LLW filter and bar plot of LLW-specific ALICE hits."""

# Load VDJdb LLWNGPMAV / HLA-A*02 reference from GitHub release (cached after first run).
vdjdb_llw = load_vdjdb_latest(
    epitope="LLWNGPMAV",
    locus="TRB",
    species="HomoSapiens",
    mhc_a_contains="A*02",
)
llw_cdr3s = {c.junction_aa for c in vdjdb_llw.clonotypes if c.junction_aa}
print(f"VDJdb A*02 LLW reference: {len(llw_cdr3s)} unique CDR3 amino acid sequences")

# Intersect each sample's ALICE hits with LLW reference.
llw_rows = []
for (donor, day), hits in sorted(yf_alice_hits.items()):
    n_llw = hits["junction_aa"].isin(llw_cdr3s).sum() if not hits.empty else 0
    llw_rows.append({"donor": donor, "day": day, "n_llw_hits": n_llw})

df_llw = pd.DataFrame(llw_rows).sort_values(["donor", "day"]).reset_index(drop=True)
display(df_llw)

# Bar plot for LLW-specific hits.
x = np.arange(len(donors_ordered))
fig, ax = plt.subplots(figsize=(9, 5))
for i, day in enumerate(days_ordered):
    sub = df_llw[df_llw["day"] == day].set_index("donor")
    heights = [sub.loc[d, "n_llw_hits"] if d in sub.index else 0 for d in donors_ordered]
    offset = (i - len(days_ordered) / 2 + 0.5) * bar_width
    ax.bar(x + offset, heights, bar_width, color=day_colors.get(day, "grey"),
           label=f"Day {day}", alpha=0.85, edgecolor="black", linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(donors_ordered)
ax.set_xlabel("Donor")
ax.set_ylabel("A*02 LLW-specific ALICE hits")
ax.set_title("ALICE hits matching HLA-A*02 LLWNGPMAV (VDJdb) per donor")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Parse AS samples

Four donors from synovial fluid CD8+ TRB repertoires.
Donor B27 status metadata is hard-coded from study design:

| Donor | B27 status |
|-------|-----------|
| 1     | B27_pos   |
| 2     | B27_pos   |
| 3     | B27_neg   |
| 4     | B27_pos   |

In [ ]:
"""Cell 6: Parse AS samples."""

AS_DONOR_META = {
    1: "B27_pos",
    2: "B27_pos",
    3: "B27_neg",
    4: "B27_pos",
}

# Expected filename pattern: Donor<N>_B27_<pos|neg>_SF_CD8.tsv.gz
as_samples: dict[int, LocusRepertoire] = {}

for donor_id, b27 in sorted(AS_DONOR_META.items()):
    pattern = f"Donor{donor_id}_B27_{b27}_SF_CD8.tsv.gz"
    fp = AS_DIR / pattern
    if not fp.exists():
        # Try globbing in case of slight filename variation.
        candidates = list(AS_DIR.glob(f"Donor{donor_id}_*.tsv.gz"))
        if not candidates:
            print(f"  WARNING: no file for Donor {donor_id} — skipping")
            continue
        fp = candidates[0]

    clonotypes = parser.parse(str(fp))
    trb = [c for c in clonotypes if c.locus == "TRB"]
    rep = filter_functional(LocusRepertoire(clonotypes=trb, locus="TRB", repertoire_id=fp.stem))
    as_samples[donor_id] = rep
    print(f"  Donor {donor_id} ({b27}): {rep.clonotype_count:,} TRB clonotypes  [{fp.name}]")

print(f"\nLoaded {len(as_samples)} AS samples")

## 7. Run ALICE on AS samples

Same ALICE parameters as YF (threshold=1, match_mode="vj", FDR < 0.05).
B27_pos donors (1, 2, 4) are expected to share convergent CDR3 neighborhoods
absent in the B27_neg donor (3).

In [ ]:
"""Cell 7: Run ALICE on all AS samples, collect FDR-filtered hits."""

as_alice_hits: dict[int, pd.DataFrame] = {}

for donor_id, rep in sorted(as_samples.items()):
    b27 = AS_DONOR_META[donor_id]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = compute_alice(
            rep,
            species="human",
            threshold=1,
            match_mode="vj",
            pgen_mode="exact",
            as_table=True,
            n_jobs=N_JOBS,
            random_seed=SEED,
        )

    tbl = result.table.copy()
    tbl["q_value"] = bh_fdr_col(tbl["p_value"].values)
    hits = tbl[tbl["q_value"] < FDR_THRESH].copy()
    hits["donor_id"] = donor_id
    hits["b27"] = b27
    as_alice_hits[donor_id] = hits
    print(f"  Donor {donor_id} ({b27}): {rep.clonotype_count:,} clonotypes → {len(hits)} ALICE hits")

# Pool for graph construction.
df_as_all_hits = pd.concat(as_alice_hits.values(), ignore_index=True)
print(f"\nTotal AS ALICE hits (with duplicates across donors): {len(df_as_all_hits)}")
print(f"Unique junction_aa in hits: {df_as_all_hits['junction_aa'].nunique()}")

## 8. Build Hamming clonotype graph for AS ALICE hits

Each ALICE hit clonotype becomes a graph node.  Edges connect pairs with
Hamming distance ≤ 1 in the same V-gene (`v_gene_match=True`).

We then build node color arrays for rendering:
- Donor 1 → orange (`#f97f31`)
- Donor 2 → blue (`#3182bd`)
- Donor 3 → green (`#31a354`) (B27_neg)
- Donor 4 → red (`#de2d26`)

In [ ]:
"""Cell 8: Build Hamming distance graph for AS ALICE hits."""
from mir.common.clonotype import Clonotype

# ── Donor color palette ────────────────────────────────────────────────────────
DONOR_COLORS = {
    1: "#f97f31",   # orange — B27_pos
    2: "#3182bd",   # blue   — B27_pos
    3: "#31a354",   # green  — B27_neg
    4: "#de2d26",   # red    — B27_pos
}

# Highlight clonotypes of special interest (TRBV9 pair from literature).
HIGHLIGHT_CLONOTYPES = {
    ("TRBV9", "CASSVGLFSTDTQYF", "TRBJ2-3"),
    ("TRBV9", "CASSVGLYSTDTQYF", "TRBJ2-3"),
}


def _as_clonotype(row: pd.Series) -> Clonotype:
    """Construct a Clonotype from a hits-table row."""
    return Clonotype(
        junction_aa=row.get("junction_aa") or "",
        v_gene=row.get("v_gene") or "",
        j_gene=row.get("j_gene") or "",
        duplicate_count=1,
        locus="TRB",
    )


# Build a list of Clonotype objects keeping track of which donor each belongs to.
# If the same clonotype appears in multiple donors, we keep one entry per donor
# (important for coloring) but deduplicate at graph level afterwards.
clonotype_list = []
clonotype_donor_ids = []

for donor_id, hits in sorted(as_alice_hits.items()):
    for _, row in hits.iterrows():
        clon = _as_clonotype(row)
        clonotype_list.append(clon)
        clonotype_donor_ids.append(donor_id)

print(f"Total nodes before deduplication: {len(clonotype_list)}")

# Build the edit-distance graph.
# v_gene_match=True: only edges between clonotypes sharing the same V gene.
g = build_edit_distance_graph(
    clonotype_list,
    metric="hamming",
    threshold=1,
    v_gene_match=True,
    n_jobs=N_JOBS,
)
g.vs["donor_id"] = clonotype_donor_ids

print(f"Graph: {g.vcount()} nodes, {g.ecount()} edges")

## 9. Render AS graph with donor colors, multi-donor pie nodes, and highlighted targets

Rendering strategy:
- Compute igraph Fruchterman-Reingold layout.
- Draw edges first with matplotlib (thin grey lines).
- For single-donor nodes: draw a filled circle using the donor color.
- For multi-donor nodes: draw a pie-chart wedge per donor.
- Overlay star markers on the two target clonotypes.

In [ ]:
"""Cell 9: Render AS Hamming graph with donor-color pie nodes and highlighted targets."""
import igraph as ig
import matplotlib.colors as mc
from matplotlib.patches import Circle, FancyArrowPatch, Wedge

# ── Layout ─────────────────────────────────────────────────────────────────────
# Keep only nodes in connected components with ≥ 2 nodes for clarity.
# If all nodes are isolated, show the full graph anyway.
comp_sizes = g.clusters().sizes()
subg = g  # default: full graph
if max(comp_sizes) > 1:
    large_comps = [i for i, sz in enumerate(comp_sizes) if sz >= 2]
    keep = [v for v in range(g.vcount()) if g.clusters().membership[v] in large_comps]
    if keep:
        subg = g.induced_subgraph(keep)

layout = subg.layout("fr", niter=1000)
coords = np.array(layout.coords)

# Normalise coordinates to [0, 1] for matplotlib.
if coords.shape[0] > 0:
    mn = coords.min(axis=0)
    mx = coords.max(axis=0)
    span = np.where(mx - mn > 0, mx - mn, 1.0)
    coords = (coords - mn) / span

NODE_RADIUS = 0.012
HIGHLIGHT_RADIUS = NODE_RADIUS * 1.5

fig, ax = plt.subplots(figsize=(14, 12))
ax.set_aspect("equal")
ax.axis("off")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)

# ── Draw edges ─────────────────────────────────────────────────────────────────
for e in subg.es:
    s, t = e.source, e.target
    xs = [coords[s, 0], coords[t, 0]]
    ys = [coords[s, 1], coords[t, 1]]
    ax.plot(xs, ys, color="#aaaaaa", linewidth=0.5, zorder=1, alpha=0.7)

# ── Group vertices by CDR3+V to find multi-donor nodes ─────────────────────────
vertex_key = [
    (subg.vs[v]["name"], subg.vs[v]["v_gene"])
    for v in range(subg.vcount())
]
# Map each node to the set of donor_ids that contributed it.
node_donor_sets: list[list[int]] = []
for v in range(subg.vcount()):
    node_donor_sets.append([subg.vs[v]["donor_id"]])

# Merge duplicate CDR3+V nodes (same junction_aa + v_gene from different donors).
# We stored one node per donor entry, so group by key.
from collections import defaultdict
key_to_nodes: dict[tuple, list[int]] = defaultdict(list)
for v, key in enumerate(vertex_key):
    key_to_nodes[key].append(v)

# ── Draw nodes ─────────────────────────────────────────────────────────────────
drawn_nodes: set[int] = set()

for key, node_ids in key_to_nodes.items():
    # Collect all donor_ids for this CDR3+V.
    donors_here = sorted({subg.vs[v]["donor_id"] for v in node_ids})
    # Representative position: mean of all duplicated node positions.
    pos = coords[node_ids, :].mean(axis=0)
    cx, cy = pos[0], pos[1]

    # Detect highlighted clonotypes.
    jaa = key[0]
    vgene = key[1]
    # Try to find j_gene for this node.
    jgene = subg.vs[node_ids[0]].get("j_gene", "")
    is_highlight = (vgene, jaa, jgene) in HIGHLIGHT_CLONOTYPES
    radius = HIGHLIGHT_RADIUS if is_highlight else NODE_RADIUS

    if len(donors_here) == 1:
        # Single donor: solid circle.
        color = DONOR_COLORS.get(donors_here[0], "#cccccc")
        circle = Circle(
            (cx, cy), radius,
            facecolor=color, edgecolor="black" if is_highlight else "white",
            linewidth=2.0 if is_highlight else 0.3,
            zorder=3,
        )
        ax.add_patch(circle)
    else:
        # Multi-donor: pie-chart wedges.
        n_d = len(donors_here)
        angle_per = 360.0 / n_d
        for k, did in enumerate(donors_here):
            wedge = Wedge(
                (cx, cy), radius,
                theta1=k * angle_per, theta2=(k + 1) * angle_per,
                facecolor=DONOR_COLORS.get(did, "#cccccc"),
                edgecolor="black" if is_highlight else "white",
                linewidth=2.0 if is_highlight else 0.3,
                zorder=3,
            )
            ax.add_patch(wedge)

    if is_highlight:
        # Overlay a star marker.
        ax.plot(
            cx, cy, marker="*", color="gold", markersize=10, zorder=5,
            markeredgecolor="black", markeredgewidth=0.5,
        )
        ax.annotate(
            jaa, (cx, cy + radius + 0.015),
            fontsize=6.5, ha="center", va="bottom", zorder=6,
            bbox=dict(boxstyle="round,pad=0.2", facecolor="lightyellow", alpha=0.85, edgecolor="grey"),
        )

    drawn_nodes.update(node_ids)

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=DONOR_COLORS[did], edgecolor="black", linewidth=0.5,
                   label=f"Donor {did} ({AS_DONOR_META[did]})")
    for did in sorted(DONOR_COLORS)
]
legend_handles.append(mpatches.Patch(facecolor="gold", edgecolor="black", linewidth=0.5,
                                      label="Highlighted target clonotypes"))
ax.legend(handles=legend_handles, loc="lower right", fontsize=9, framealpha=0.9)

ax.set_title(
    f"AS ALICE hits — Hamming-1 clonotype graph  ({subg.vcount()} nodes, {subg.ecount()} edges)\n"
    "Node color = donor | Multi-donor = pie | Star = target CDR3",
    fontsize=11,
)
plt.tight_layout()
plt.show()